# HCMC Urban Planning Land-Use Crawler

**Pipeline**
1. Read a CSV that contains `Latitude` and `Longitude` columns (WGS84, from Google Maps).
2. Call the HCMC planning API (`thongtinquyhoach.hochiminhcity.gov.vn`) to retrieve the land-use zone for each coordinate.
3. If the origin point falls entirely on a traffic zone, automatically probe 8 neighbouring offsets (~5.5 m) and use the first result that contains a non-traffic zone.
4. Write an enriched CSV with all planning (`QHPK`) columns appended.

**Environment setup** — copy `.env.example` to `.env` and fill in the paths before running:
```
INPUT_CSV_PATH=path/to/input.csv
OUTPUT_CSV_PATH=path/to/output.csv
```

## 0 · Environment & Configuration

In [10]:
import ast
import csv
import json
import os
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env from the working directory

# ── File paths (set via .env or override here) ───────────────
INPUT_CSV_PATH = os.getenv("INPUT_CSV_PATH", "input.csv")
OUTPUT_CSV_PATH = os.getenv("OUTPUT_CSV_PATH", "output_quyhoach.csv")

# ── API / crawler settings ───────────────────────────────────
# Lấy từ file .env để bảo mật và dễ cấu hình
API_URL = os.getenv("API_URL")
API_HOST = os.getenv("API_HOST")
ORIGIN_URL = os.getenv("ORIGIN_URL")

# Kiểm tra để tránh lỗi nếu quên khai báo trong .env
if not API_URL or not API_HOST or not ORIGIN_URL:
    raise ValueError("Missing API configuration. Please check your .env file.")

MAX_RETRY = int(os.getenv("MAX_RETRY", 3))
BACKOFF_FACTOR = int(os.getenv("BACKOFF_FACTOR", 3))     # seconds per retry increment
ROW_DELAY = float(os.getenv("ROW_DELAY", 1.0))           # seconds between rows
NEIGHBOUR_DELAY = 1.5                                    # seconds between neighbour probes
OFFSET_DEGREE = 0.00005                                  # ~5.5 m offset for neighbour probes

# Statuses that indicate a permanent failure — skip on retry runs
PERMANENT_ERROR_KEYWORDS = [
    "server trả về rỗng",
    "bị khóa: the land is temporarily locked!",
]

# Separator used to join multiple planning zones inside one cell
ZONE_SEP = " | "

# Output column names for the planning data
PLANNING_COLUMNS = [
    "District",          # Quận / Huyện
    "Ward",              # Phường / Xã
    "Parcel_No",         # Số thửa
    "Map_No",            # Số tờ
    "Parcel_Area_m2",    # Diện tích lô (m²)
    "Plan_Name",         # Tên đồ án quy hoạch
    "Zone_Count",        # Số ô quy hoạch trả về
    "Zone_Function",     # Chức năng từng ô, phân cách bằng ZONE_SEP
    "Zone_Area_m2",      # Diện tích từng ô (m²)
    "Zone_Area_Pct",     # % diện tích từng ô
    "Zone_Code",         # Mã quy ước (maquyuoc)
    "Zone_Block_Code",   # Mã ô phố (maopho)
    "Status",
]

## 1 · HTTP Session

In [11]:
def build_session() -> requests.Session:
    """Create a persistent HTTP session with headers that mimic a real browser."""
    session = requests.Session()
    session.headers.update({
        "Host": API_HOST,
        "Accept": "application/json, text/plain, */*",
        "Content-Type": "application/x-www-form-urlencoded",
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/146.0.0.0 Safari/537.36"
        ),
        "Origin": ORIGIN_URL,
        "Referer": ORIGIN_URL + "/",
    })
    return session


SESSION = build_session()

## 2 · API Helpers

In [12]:
def _call_api(lat: float, lon: float) -> tuple:
    """
    POST a single coordinate to the planning API.

    Returns
    -------
    (result_data, None)          on success
    (None, error_message: str)   on failure
    """
    payload = {"Lat": lat, "Lon": lon}

    for attempt in range(1, MAX_RETRY + 1):
        try:
            if attempt > 1:
                time.sleep(BACKOFF_FACTOR * (attempt - 1))

            response = SESSION.post(API_URL, data=payload, timeout=15)

            if response.status_code != 200:
                return None, f"HTTP {response.status_code}"

            raw = response.text.strip() if response.text else ""
            if not raw:
                return None, "server trả về rỗng"

            try:
                data = response.json()
            except (json.JSONDecodeError, ValueError):
                try:
                    data = ast.literal_eval(raw)
                except Exception:
                    return None, f"Parse error: {raw[:50]}"

            if isinstance(data, dict) and "error" in data:
                return None, f"bị khóa: {data['error']}"
            if data.get("blocked") == 1:
                return None, "IP blocked"

            return data, None

        except requests.exceptions.Timeout:
            if attempt == MAX_RETRY:
                return None, "Timeout"
        except requests.exceptions.RequestException as exc:
            if attempt == MAX_RETRY:
                return None, f"Network error: {str(exc)[:50]}"
        except Exception as exc:
            return None, f"Unexpected error: {str(exc)[:50]}"

    return None, "Max retries exceeded"


def _parse_response(raw_data: dict) -> dict:
    """
    Convert the raw API dict into a flat record aligned with PLANNING_COLUMNS.
    Multiple planning zones are joined with ZONE_SEP.
    """
    record = {col: "" for col in PLANNING_COLUMNS}
    record["Zone_Count"] = 0
    record["Status"] = "OK"

    # ── General parcel info ──────────────────────────────────
    raw_general = raw_data.get("ThongTinChung", "")
    if raw_general and raw_general not in ("[]", "{}"):
        try:
            general = (
                json.loads(raw_general)
                if isinstance(raw_general, str)
                else raw_general
            )
            if isinstance(general, dict):
                record["District"] = general.get("tenquanhuyen", "")
                record["Ward"] = general.get("tenphuongxa", "")
                record["Parcel_No"] = general.get("sothua", "")
                record["Map_No"] = general.get("soto", "")
                record["Parcel_Area_m2"] = general.get("dientich", "")
                plans = general.get("dsdoan", [])
                if isinstance(plans, list):
                    record["Plan_Name"] = ZONE_SEP.join(plans)
        except Exception:
            record["Status"] = "Parse error: ThongTinChung"

    # ── Zone (QHPK) details ──────────────────────────────────
    raw_zones = raw_data.get("QHPK", "")
    if raw_zones and raw_zones not in ("[]", "{}"):
        try:
            zones = (
                json.loads(raw_zones)
                if isinstance(raw_zones, str)
                else raw_zones
            )
            if isinstance(zones, dict):
                zones = [zones]

            if isinstance(zones, list) and zones:
                functions, areas, pcts, codes, blocks = [], [], [], [], []

                for zone in zones:
                    props = zone.get("properties", {}) if isinstance(zone, dict) else {}

                    function = props.get("chucnang") or props.get("chucNang") or ""
                    area = props.get("dientich", "")
                    pct = props.get("tldientich", "")
                    code = props.get("maquyuoc", "") or ""
                    block = props.get("maopho", "") or ""

                    try:
                        area = f"{float(area):.2f}"
                    except (TypeError, ValueError):
                        area = str(area)

                    try:
                        pct = f"{float(pct):.2f}%"
                    except (TypeError, ValueError):
                        pct = str(pct)

                    functions.append(function)
                    areas.append(area)
                    pcts.append(pct)
                    codes.append(str(code))
                    blocks.append(str(block))

                record["Zone_Count"] = len(zones)
                record["Zone_Function"] = ZONE_SEP.join(functions)
                record["Zone_Area_m2"] = ZONE_SEP.join(areas)
                record["Zone_Area_Pct"] = ZONE_SEP.join(pcts)
                record["Zone_Code"] = ZONE_SEP.join(codes)
                record["Zone_Block_Code"] = ZONE_SEP.join(blocks)

        except Exception as exc:
            record["Status"] = f"Parse error QHPK: {str(exc)[:60]}"

    return record


def _has_non_traffic_zone(zone_function: str) -> bool:
    """Return True if at least one zone is NOT a traffic/road zone."""
    zones = [z.strip().lower() for z in zone_function.split("|")]
    return any("giao thông" not in z for z in zones if z)


def fetch_planning_info(lat: float, lon: float) -> dict:
    """
    Fetch planning data for a coordinate.

    If the origin point returns only traffic zones, probe 8 neighbouring
    offsets (~5.5 m) and return the first result that has a non-traffic zone.
    Falls back to the origin result if all neighbours are traffic-only.
    """
    neighbour_offsets = [
        (0.0, 0.0),
        (OFFSET_DEGREE, 0.0), (-OFFSET_DEGREE, 0.0),
        (0.0, OFFSET_DEGREE), (0.0, -OFFSET_DEGREE),
        (OFFSET_DEGREE, OFFSET_DEGREE), (-OFFSET_DEGREE, -OFFSET_DEGREE),
        (OFFSET_DEGREE, -OFFSET_DEGREE), (-OFFSET_DEGREE, OFFSET_DEGREE),
    ]

    origin_record = None

    for d_lat, d_lon in neighbour_offsets:
        is_origin = d_lat == 0.0 and d_lon == 0.0
        if not is_origin:
            time.sleep(NEIGHBOUR_DELAY)

        data, error = _call_api(lat + d_lat, lon + d_lon)

        if error:
            record = {col: "" for col in PLANNING_COLUMNS}
            record["Status"] = error
            # Hard errors (IP block / network) — abort immediately
            if any(k in error for k in ("blocked", "Timeout", "Network error")):
                return origin_record or record
            if is_origin:
                origin_record = record
            continue

        record = _parse_response(data)
        has_non_traffic = _has_non_traffic_zone(record["Zone_Function"])

        if is_origin:
            origin_record = record
            # Origin already has useful zones — no need to probe neighbours
            if has_non_traffic or not record["Zone_Function"]:
                return record
        else:
            if has_non_traffic:
                record["Status"] = "OK (auto-shifted: origin was traffic-only)"
                return record

    return origin_record

## 3 · Data Assertions (Sanity Checks)

In [13]:
def assert_input_file(path: str) -> list[str]:
    """
    Validate the input CSV before processing.
    Raises ValueError with a descriptive message on failure.
    Returns the list of field names on success.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Input file not found: {path}")

    with open(path, encoding="utf-8-sig") as fh:
        reader = csv.DictReader(fh)
        fieldnames = reader.fieldnames

    if not fieldnames:
        raise ValueError("Input CSV has no header row.")

    required = {"Latitude", "Longitude"}
    missing = required - set(fieldnames)
    if missing:
        raise ValueError(f"Input CSV is missing required columns: {missing}")

    return list(fieldnames)


def assert_planning_record(record: dict, row_index: int) -> None:
    """
    Lightweight sanity check on a parsed planning record.
    Prints a warning (does not raise) so the pipeline continues.
    """
    if record.get("Zone_Count", 0) == 0 and record.get("Status") == "OK":
        print(f"  [WARN] Row {row_index}: Status is OK but Zone_Count is 0.")

    zone_count = record.get("Zone_Count", 0)
    function_parts = [p for p in record.get("Zone_Function", "").split("|") if p.strip()]
    if isinstance(zone_count, int) and zone_count != len(function_parts) and zone_count > 0:
        print(
            f"  [WARN] Row {row_index}: Zone_Count={zone_count} "
            f"but Zone_Function has {len(function_parts)} parts."
        )

## 4 · Pipeline — Full Crawl

In [14]:
def run_full_crawl(input_path: str, output_path: str) -> None:
    """
    Read every row from *input_path*, fetch its planning data, and write
    the enriched rows to *output_path*.
    """
    fieldnames = assert_input_file(input_path)

    with open(input_path, encoding="utf-8-sig") as fh:
        total_rows = sum(1 for _ in csv.DictReader(fh))

    all_columns = fieldnames + PLANNING_COLUMNS
    print(f"Starting full crawl: {total_rows} rows → {output_path}\n")

    with (
        open(input_path, encoding="utf-8-sig") as f_in,
        open(output_path, mode="w", encoding="utf-8-sig", newline="") as f_out,
    ):
        reader = csv.DictReader(f_in)
        writer = csv.DictWriter(f_out, fieldnames=all_columns)
        writer.writeheader()

        for idx, row in enumerate(reader, 1):
            lat = row.get("Latitude", "").strip()
            lon = row.get("Longitude", "").strip()
            link = row.get("Link", "N/A")

            print(f"[{idx}/{total_rows}] {link[:50]} | ({lat}, {lon})")

            if lat and lon:
                record = fetch_planning_info(float(lat), float(lon))
                assert_planning_record(record, idx)
                row.update(record)
                print(
                    f"  zones={record['Zone_Count']} | "
                    f"{record['Zone_Function']} | "
                    f"status={record['Status']}"
                )
            else:
                row["Status"] = "Missing coordinates"
                print("  [SKIP] Missing coordinates")

            writer.writerow(row)
            time.sleep(ROW_DELAY)

    print(f"\nDone. Output saved to: {output_path}")

## 5 · Pipeline — Retry Failed Rows

In [15]:
def _needs_retry(row: dict) -> bool:
    """
    Return True if a row should be re-fetched.

    Criteria:
    - Has valid coordinates.
    - District is empty (no data was retrieved).
    - Status does not indicate a permanent / unrecoverable failure.
    """
    if not row.get("Latitude", "").strip() or not row.get("Longitude", "").strip():
        return False
    if row.get("District", "").strip():
        return False
    status = row.get("Status", "").strip().lower()
    return not any(keyword in status for keyword in PERMANENT_ERROR_KEYWORDS)


def run_retry_crawl(input_path: str, output_path: str) -> None:
    """
    Re-fetch only the rows that previously failed and are worth retrying.
    Reads *input_path* into memory, updates failed rows in-place, then
    writes the full dataset to *output_path* (which may be the same file).
    """
    assert_input_file(input_path)

    with open(input_path, encoding="utf-8-sig") as fh:
        reader = csv.DictReader(fh)
        fieldnames = reader.fieldnames
        all_rows = list(reader)

    retry_indices = [i for i, row in enumerate(all_rows) if _needs_retry(row)]

    total = len(all_rows)
    n_ok = total - len(retry_indices)
    n_retry = len(retry_indices)

    print(f"Total rows    : {total}")
    print(f"Already OK    : {n_ok}")
    print(f"To retry      : {n_retry}")

    if n_retry == 0:
        print("\nNothing to retry — dataset is complete.")
        return

    print(f"\nStarting retry for {n_retry} rows...\n")
    success_count = 0
    still_failing = 0

    for order, i in enumerate(retry_indices, 1):
        row = all_rows[i]
        lat = row["Latitude"].strip()
        lon = row["Longitude"].strip()
        link = row.get("Link", "N/A")
        old_status = row.get("Status", "").strip()

        print(f"[{order}/{n_retry}] row #{i + 1} | {link[:45]}")
        print(f"  prev_status: {old_status}")

        record = fetch_planning_info(float(lat), float(lon))
        assert_planning_record(record, i + 1)
        all_rows[i].update(record)

        if record.get("District", "").strip():
            success_count += 1
            print(f"  [OK] zones={record['Zone_Count']} | {record['Zone_Function']}")
        else:
            still_failing += 1
            print(f"  [FAIL] status={record['Status']}")

        time.sleep(ROW_DELAY)

    with open(output_path, mode="w", encoding="utf-8-sig", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n{'=' * 50}")
    print(f"Retry complete. Output saved to: {output_path}")
    print(f"  Recovered : {success_count}")
    print(f"  Still failing : {still_failing}")
    if still_failing:
        print("  → Re-run this cell to attempt another retry pass.")

## 6 · Run

In [16]:
# ── Cell 6a: Full crawl (first-time run) ─────────────────────
# run_full_crawl(INPUT_CSV_PATH, OUTPUT_CSV_PATH)

# ── Cell 6b: Retry failed rows (subsequent runs) ─────────────
# run_retry_crawl(OUTPUT_CSV_PATH, OUTPUT_CSV_PATH)

# Uncomment the appropriate call above, then run this cell.
print("Ready. Uncomment one of the lines above and run this cell.")

Ready. Uncomment one of the lines above and run this cell.
